In [7]:
import numpy as np
import pandas as pd
import time
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =================================================================
# 1. LOAD & PREPROCESSING DATA
# =================================================================
df = pd.read_csv('fix.csv')
data = df[['title','authors','categories','description']].copy()

# Handle missing values
data = data.fillna('')

# Gabungkan kolom teks
data['combined'] = data['title'] + " " + data['authors'] + " " + data['categories'] + " " + data['description']
data['combined'] = data['combined'].str.lower()

# Tokenisasi & Stopwords
data['tokens'] = data['combined'].apply(lambda x: x.split())
stopwords = ['and','a','about','the','of','is','that']
data['filtered'] = data['tokens'].apply(lambda x: [w for w in x if w not in stopwords])

# Stemming
stemmer = PorterStemmer()
data['stemmed'] = data['filtered'].apply(lambda x: [stemmer.stem(word) for word in x])
data['final'] = data['stemmed'].apply(lambda x: ' '.join(x))


In [8]:
# =================================================================
# 2. SPLIT DATA: HOLD-OUT 3 ARAH (TRAIN : VALIDATION : TEST)
# =================================================================
# Langkah 1: Ambil 10% untuk Test Data (Unseen Data)
train_val_data, test_data = train_test_split(data, test_size=0.10, random_state=42)

# Langkah 2: Sisa 90% dipecah lagi -> 80% Train, 10% Validation
train_data, val_data = train_test_split(train_val_data, test_size=0.1111, random_state=42)

# Reset Index agar rapi
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print("--- Hasil Split Data (Hold-out 3 Arah) ---")
print(f"Total Data Awal : {len(data)}")
print(f"Data Train (80%) : {len(train_data)}")
print(f"Data Val   (10%) : {len(val_data)}")
print(f"Data Test  (10%) : {len(test_data)}\n")


--- Hasil Split Data (Hold-out 3 Arah) ---
Total Data Awal : 6810
Data Train (80%) : 5448
Data Val   (10%) : 681
Data Test  (10%) : 681



In [9]:
# =================================================================
# 3. TF-IDF & COSINE SIMILARITY
# =================================================================
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    min_df=2,
    max_df=0.8
)

# Fit hanya pada Train Data, lalu transform ke semuanya
tfidf_train = tfidf.fit_transform(train_data['final'])
tfidf_val = tfidf.transform(val_data['final'])
tfidf_test = tfidf.transform(test_data['final'])

# Hitung Cosine Similarity antara Test Data (Unseen) terhadap Train Data
cosine_sim_test = cosine_similarity(tfidf_test, tfidf_train)



In [10]:
# =================================================================
# 4. FUNGSI REKOMENDASI (Testing manual)
# =================================================================
def rekomendasi(judul):
    judul = str(judul).strip().lower()
    
    test_data['title_clean'] = test_data['title'].astype(str).str.lower()
    hasil = test_data[test_data['title_clean'].str.contains(judul, na=False)]
    
    if hasil.empty:
        print("Judul tidak ditemukan di Data Uji (Test Data)")
        return
    
    idx = hasil.index[0]
    
    skor = list(enumerate(cosine_sim_test[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    
    print("Rekomendasi untuk:", test_data.loc[idx, 'title'])
    print("-" * 30)
    
    for count, i in enumerate(skor):
        print(f"{train_data.loc[i[0], 'title']} -> {round(i[1], 3)}")
        if count == 2: # Ambil top 3
            break


In [11]:
# =================================================================
# 5. FUNGSI UJI VALIDITAS / EVALUASI (OPTIMAL & CEPAT)
# =================================================================
def hitung_metrik(idx, cosine_sim_row, train_df, test_df, k=3):
    # Mengambil k rekomendasi teratas pakai Numpy (jauh lebih cepat)
    rekom_idx = np.argsort(cosine_sim_row)[-k:][::-1]

    kategori_asli = set(str(test_df.loc[idx, 'categories']).lower().split())

    relevansi = []
    relevan_count = 0

    for i in rekom_idx:
        kategori_rekom = set(str(train_df.loc[i, 'categories']).lower().split())
        if len(kategori_asli & kategori_rekom) > 0:
            relevansi.append(1)
            relevan_count += 1
        else:
            relevansi.append(0)

    # Precision
    precision_val = relevan_count / k

    # NDCG
    dcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevansi)])
    ideal = sorted(relevansi, reverse=True)
    idcg_val = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal)])
    ndcg_val = 0 if idcg_val == 0 else dcg_val / idcg_val

    return precision_val, ndcg_val


def evaluasi_sistem(train_df, test_df, cosine_sim_matrix, k=3):
    start_time = time.time()   
    total_precision = 0
    total_ndcg = 0

    # Evaluasi menggunakan Unseen Data (Test Data)
    for idx in range(len(test_df)):
        prec, ndcg = hitung_metrik(idx, cosine_sim_matrix[idx], train_df, test_df, k)
        total_precision += prec
        total_ndcg += ndcg

    avg_precision = total_precision / len(test_df)
    avg_ndcg = total_ndcg / len(test_df)

    running_time = time.time() - start_time

    print(f"Precision@{k} : {round(avg_precision, 3)}")
    print(f"NDCG@{k}      : {round(avg_ndcg, 3)}")
    print(f"Running Time : {round(running_time, 4)} detik")
    print("-" * 30) 



In [16]:
# =================================================================
# 6. PEMANGGILAN FUNGSI (EKSEKUSI)
# =================================================================
rekomendasi("The Princess of the Chalet School")

print("\n--- HASIL EVALUASI PADA UNSEEN DATA (TEST DATA) ---")
evaluasi_sistem(train_data, test_data, cosine_sim_test, k=3)
evaluasi_sistem(train_data, test_data, cosine_sim_test, k=5)
evaluasi_sistem(train_data, test_data, cosine_sim_test, k=10)

Rekomendasi untuk: The Princess of the Chalet School
------------------------------
The Princess Diaries -> 0.251
Mary Anne and the Little Princess -> 0.18
Heathersleigh Homecoming -> 0.167

--- HASIL EVALUASI PADA UNSEEN DATA (TEST DATA) ---
Precision@3 : 0.524
NDCG@3      : 0.656
Running Time : 0.197 detik
------------------------------
Precision@5 : 0.519
NDCG@5      : 0.681
Running Time : 0.1563 detik
------------------------------
Precision@10 : 0.499
NDCG@10      : 0.703
Running Time : 0.1705 detik
------------------------------


In [17]:
# Menampilkan 10 judul buku acak yang ada di dalam Test Data
print(test_data['title'].sample(20).to_string())

301                     The Treasure Principle
393                       Dirty Jokes and Beer
42          Kaddish and Other Poems: 1958-1960
340                     Happy Birthday to You!
517                         The Caves of Steel
655                          The Call of Earth
397                                 Sebastopol
54                             Fermat's Enigma
464                         Bad Boys Over Easy
377                          Ten Short Stories
195                   The World of Jules Verne
577             The Oedipus Plays of Sophocles
84                                 Texas! Sage
471            The Good Husband of Zebra Drive
412                            Midnight Riders
621                                   Stargirl
6                              Song of Solomon
240    Notebook of a Return to the Native Land
235                                    Tsubasa
160                               The Tiny One
